In [ ]:
from pathlib import Path

import numpy as np

from nicht_riemann_data.transforms import spacings
from nicht_riemann_data.diagnostics import describe


DATA_FILE = Path("data/raw/zeros1")

assert DATA_FILE.exists(), f"Missing data file: {DATA_FILE}"

print("data:", DATA_FILE)
print("numpy:", np.__version__)
print("OK")

In [ ]:
gamma = np.loadtxt(DATA_FILE, dtype=np.float64)

print("N:", len(gamma))
print("first:", gamma[:5])
print("last:", gamma[-5:])

In [ ]:
assert gamma.ndim == 1
assert gamma.dtype == np.float64
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)

print("shape:", gamma.shape)
print("dtype:", gamma.dtype)
print("finite:", np.all(np.isfinite(gamma)))
print("strictly increasing:", np.all(np.diff(gamma) > 0))
print("OK")

In [ ]:
delta = spacings(gamma)

assert delta.shape == (len(gamma) - 1,)
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)

print("zeros:", len(gamma))
print("spacings:", len(delta))
print("first spacings:", delta[:10])

In [ ]:
print("gamma:")
print(describe(gamma))

print("\ndelta:")
print(describe(delta))

In [ ]:
gamma_range = gamma[-1] - gamma[0]
mean_spacing = np.mean(delta)
range_per_spacing = gamma_range / len(delta)

print("range:", gamma_range)
print("mean spacing:", mean_spacing)
print("range / number of spacings:", range_per_spacing)

assert np.isclose(mean_spacing, range_per_spacing)

In [ ]:
percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]
percentiles = np.percentile(delta, percentile_levels)

for p, value in zip(percentile_levels, percentiles):
    print(f"{p:>3}% : {value:.12f}")

In [ ]:
BLOCK_SIZE = 1000

num_blocks = len(delta) // BLOCK_SIZE

block_means = np.array([
    np.mean(delta[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE])
    for i in range(num_blocks)
])

remainder_delta = delta[num_blocks * BLOCK_SIZE:]

print("block size:", BLOCK_SIZE)
print("full blocks:", num_blocks)
print("remainder:", len(remainder_delta))

print("\nfirst block means:")
print(block_means[:10])

print("\nlast block means:")
print(block_means[-10:])

In [ ]:
print("local mean spacing:")
print("  min :", block_means.min())
print("  max :", block_means.max())
print("  mean:", block_means.mean())
print("  std :", block_means.std())

print("\nratio max/min:", block_means.max() / block_means.min())

In [ ]:
local_residuals = np.empty_like(delta)

for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE

    local_residuals[start:end] = delta[start:end] - block_means[i]

if len(remainder_delta):
    start = num_blocks * BLOCK_SIZE
    local_residuals[start:] = remainder_delta - np.mean(remainder_delta)

assert local_residuals.shape == delta.shape

print("local-mean residual:")
print("  std :", np.std(local_residuals))
print("  min :", np.min(local_residuals))
print("  max :", np.max(local_residuals))

In [ ]:
predicted_global = gamma[0] + np.arange(len(gamma)) * mean_spacing
residual_global = gamma - predicted_global

print("global constant-spacing baseline:")
print("  residual std:", np.std(residual_global))
print("  residual min:", np.min(residual_global))
print("  residual max:", np.max(residual_global))

In [ ]:
predicted_delta = np.empty_like(delta)

for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE

    predicted_delta[start:end] = block_means[i]

if len(remainder_delta):
    predicted_delta[num_blocks * BLOCK_SIZE:] = np.mean(remainder_delta)

residual = delta - predicted_delta

assert predicted_delta.shape == delta.shape
assert residual.shape == delta.shape

print("local block baseline:")
print("  residual std:", np.std(residual))
print("  residual min:", np.min(residual))
print("  residual max:", np.max(residual))